# YOLO11-seg

- Dorian Ticona

## 1. Instalar dependencias

Ejecuta esta celda una sola vez por runtime.

In [ ]:
%%capture
%pip install -q -U ultralytics
%pip install -q "pandas==2.2.2" "pillow==11.3.0" "tqdm>=4.67.0" opencv-python

## 2. Imports y verificación del entorno

In [ ]:
from __future__ import annotations

import base64
import platform
import subprocess
import time
from pathlib import Path
from typing import Optional, Union

import cv2
import pandas as pd
import torch
from IPython.display import HTML, display
from tqdm.auto import tqdm
from ultralytics import YOLO

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


## 3. Configuración general

In [ ]:
OUTPUT_DIR = Path("/content/yolo11_seg_static_video_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE: Union[int, str] = 0 if torch.cuda.is_available() else "cpu"

MODEL = 0 # nano, small, medium
IMG_SIZE = 1280
CONF = 0.35
IOU = 0.70
RETINA_MASKS = True
FRAME_STRIDE = 1
MAX_FRAMES: Optional[int] = None
USE_TRACKING = True
TRACKER = "bytetrack.yaml"  # alternativa: "botsort.yaml"

In [ ]:
import numpy as np
from typing import Literal

SEGMENTATION_MODE: Literal[
    "instance",
    "class",
    "binary",
    "contour",
    "cutout",
    "blur_background",
] = "instance"

MASK_ALPHA = 0.45
CONTOUR_THICKNESS = 2

def deterministic_color(index: int) -> tuple[int, int, int]:
    rng = np.random.default_rng(seed=index + 12345)
    color = rng.integers(40, 256, size=3)
    return int(color[0]), int(color[1]), int(color[2])

def get_track_ids(result, fallback_count: int) -> np.ndarray:
    if (
        result.boxes is not None
        and hasattr(result.boxes, "id")
        and result.boxes.id is not None
    ):
        return result.boxes.id.detach().cpu().numpy().astype(int)

    return np.arange(fallback_count)


def get_result_masks_original_size(result, frame_bgr: np.ndarray) -> list[np.ndarray]:
    if result.masks is None or result.masks.data is None:
        return []

    frame_h, frame_w = frame_bgr.shape[:2]
    masks_np = result.masks.data.detach().cpu().numpy()

    resized_masks: list[np.ndarray] = []

    for mask in masks_np:
        if mask.shape[:2] != (frame_h, frame_w):
            mask = cv2.resize(
                mask,
                (frame_w, frame_h),
                interpolation=cv2.INTER_NEAREST,
            )

        resized_masks.append(mask > 0.5)

    return resized_masks


def plot_segmentation_mode(
    result,
    frame_bgr: np.ndarray,
    mode: str = SEGMENTATION_MODE,
    alpha: float = MASK_ALPHA,
    contour_thickness: int = CONTOUR_THICKNESS,
) -> np.ndarray:
    masks = get_result_masks_original_size(result, frame_bgr)

    if not masks:
        return frame_bgr.copy()

    annotated = frame_bgr.copy()
    overlay = frame_bgr.copy()

    if result.boxes is not None and len(result.boxes) > 0:
        class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
    else:
        class_ids = np.zeros(len(masks), dtype=int)

    if mode == "instance":
        track_ids = get_track_ids(result, fallback_count=len(masks))

        for mask, track_id in zip(masks, track_ids):
            color = deterministic_color(int(track_id))
            overlay[mask] = color

        annotated = cv2.addWeighted(overlay, alpha, frame_bgr, 1 - alpha, 0)
        return annotated

    if mode == "class":
        for mask, class_id in zip(masks, class_ids):
            color = deterministic_color(int(class_id))
            overlay[mask] = color

        annotated = cv2.addWeighted(overlay, alpha, frame_bgr, 1 - alpha, 0)
        return annotated

    if mode == "binary":
        combined_mask = np.zeros(frame_bgr.shape[:2], dtype=bool)

        for mask in masks:
            combined_mask |= mask

        overlay[combined_mask] = (0, 255, 0)
        annotated = cv2.addWeighted(overlay, alpha, frame_bgr, 1 - alpha, 0)
        return annotated

    if mode == "contour":
        annotated = frame_bgr.copy()

        for instance_index, mask in enumerate(masks):
            color = deterministic_color(instance_index)
            mask_uint8 = (mask.astype(np.uint8) * 255)

            contours, _ = cv2.findContours(
                mask_uint8,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE,
            )

            cv2.drawContours(
                annotated,
                contours,
                contourIdx=-1,
                color=color,
                thickness=contour_thickness,
            )

        return annotated

    if mode == "cutout":
        combined_mask = np.zeros(frame_bgr.shape[:2], dtype=bool)

        for mask in masks:
            combined_mask |= mask

        dark_background = (frame_bgr * 0.18).astype(np.uint8)
        annotated = dark_background.copy()
        annotated[combined_mask] = frame_bgr[combined_mask]
        return annotated

    if mode == "blur_background":
        combined_mask = np.zeros(frame_bgr.shape[:2], dtype=bool)

        for mask in masks:
            combined_mask |= mask

        blurred = cv2.GaussianBlur(frame_bgr, (41, 41), 0)
        annotated = blurred.copy()
        annotated[combined_mask] = frame_bgr[combined_mask]
        return annotated

    raise ValueError(
        f"SEGMENTATION_MODE inválido: {mode}. "
        "Usa: instance, class, binary, contour, cutout o blur_background."
    )

# 4. Tres variantes de instanciación YOLO11-seg

Ejecuta **solo una** de las tres opciones principales.

- **Nano**: más rápido.
- **Small**: mejor balance.
- **Medium**: más precisión, más lento.

In [ ]:
YOLO11_SEG = ["yolo11n-seg.pt", "yolo11s-seg.pt", "yolo11m-seg.pt"]

model = YOLO(YOLO11_SEG[MODEL])
MODEL_NAME = YOLO11_SEG[MODEL].replace(".pt", "")

print("Modelo seleccionado:", MODEL_NAME)

Modelo seleccionado: yolo11n-seg


## 5. Subir video a Colab

Ejecuta esta celda y selecciona tu video.

In [ ]:
from google.colab import files

uploaded = files.upload()

VIDEO_PATH = Path(next(iter(uploaded.keys())))
print("Video subido:", VIDEO_PATH)

Saving 27091-361827476_medium.mp4 to 27091-361827476_medium.mp4
Video subido: 27091-361827476_medium.mp4


## 6. Inspeccionar metadata del video

In [ ]:
def inspect_video(video_path: Union[str, Path]) -> dict:
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    metadata = {
        "path": str(video_path),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": fps,
        "frames": frames,
        "duration_seconds": frames / fps if fps > 0 else None,
    }

    cap.release()
    return metadata


metadata = inspect_video(VIDEO_PATH)

print("Video:", metadata["path"])
print("Resolución:", f'{metadata["width"]}x{metadata["height"]}')
print("FPS:", metadata["fps"])
print("Frames:", metadata["frames"])
print("Duración aprox. segundos:", metadata["duration_seconds"])

Video: 27091-361827476_medium.mp4
Resolución: 1280x720
FPS: 25.0
Frames: 311
Duración aprox. segundos: 12.44


## 8. Helpers para convertir, mostrar y descargar el video

OpenCV guarda MP4 con codec `mp4v`. Para verlo bien se convierte a H.264 con `ffmpeg`.

In [ ]:
def convert_to_browser_mp4(input_path: Union[str, Path], output_path: Union[str, Path]) -> Path:
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    command = [
        "ffmpeg",
        "-y",
        "-i", str(input_path),
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-preset", "veryfast",
        "-crf", "23",
        str(output_path),
    ]

    subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return output_path


def display_video(video_path: Union[str, Path], width: int = 720) -> None:
    video_path = Path(video_path)

    with video_path.open("rb") as f:
        video_bytes = f.read()

    encoded = base64.b64encode(video_bytes).decode("ascii")

    display(HTML(f'''
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    '''))


def download_file(path: Union[str, Path]) -> None:
    from google.colab import files
    files.download(str(path))

# 9. Procesar video con YOLO11-seg

In [ ]:
def process_yolo11_seg_video(
    model: YOLO,
    video_path: Union[str, Path],
    output_path: Union[str, Path],
    imgsz: int = IMG_SIZE,
    conf: float = CONF,
    iou: float = IOU,
    retina_masks: bool = RETINA_MASKS,
    device: Union[int, str] = DEVICE,
    frame_stride: int = FRAME_STRIDE,
    max_frames: Optional[int] = MAX_FRAMES,
    draw_info: bool = True,
    collect_stats: bool = True,
) -> tuple[Path, pd.DataFrame]:
    video_path = Path(video_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if frame_stride < 1:
        raise ValueError("frame_stride debe ser >= 1.")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = float(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fps_out = fps_in if fps_in and fps_in > 1 else 24.0
    effective_fps = fps_out / frame_stride

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, effective_fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"No se pudo crear el video de salida: {output_path}")

    progress_total = total_frames if max_frames is None else min(total_frames, max_frames)
    progress = tqdm(total=progress_total, desc=f"Procesando {video_path.name}")

    stats_rows: list[dict] = []
    read_frames = 0
    processed_frames = 0
    t_global = time.perf_counter()

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if max_frames is not None and read_frames >= max_frames:
                break

            progress.update(1)

            current_frame_index = read_frames
            read_frames += 1

            if current_frame_index % frame_stride != 0:
                continue

            t0 = time.perf_counter()

            if USE_TRACKING:
                results = model.track(
                    source=frame,
                    persist=True,
                    tracker=TRACKER,
                    imgsz=imgsz,
                    conf=conf,
                    iou=iou,
                    retina_masks=retina_masks,
                    device=device,
                    verbose=False,
                )
            else:
                results = model.predict(
                    source=frame,
                    imgsz=imgsz,
                    conf=conf,
                    iou=iou,
                    retina_masks=retina_masks,
                    device=device,
                    verbose=False,
                )

            result = results[0]

            annotated = plot_segmentation_mode(
                result=result,
                frame_bgr=frame,
                mode=SEGMENTATION_MODE,
                alpha=MASK_ALPHA,
                contour_thickness=CONTOUR_THICKNESS,
            )

            inference_dt = time.perf_counter() - t0
            model_fps = 1.0 / max(inference_dt, 1e-9)

            if draw_info:
                text = f"{MODEL_NAME} | imgsz={imgsz} | retina={retina_masks} | FPS={model_fps:.1f}"
                cv2.putText(
                    annotated,
                    text,
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.85,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )

            writer.write(annotated)
            processed_frames += 1

            if collect_stats and result.boxes is not None and len(result.boxes) > 0:
                classes = result.boxes.cls.detach().cpu().numpy().astype(int)
                confs = result.boxes.conf.detach().cpu().numpy()
                names = result.names
                mask_available = result.masks is not None and result.masks.data is not None

                for class_id in sorted(set(classes.tolist())):
                    class_mask = classes == class_id
                    stats_rows.append({
                        "model": MODEL_NAME,
                        "source_video": str(video_path),
                        "read_frame_index": current_frame_index,
                        "processed_frame_index": processed_frames - 1,
                        "class_id": int(class_id),
                        "class_name": names[int(class_id)],
                        "count": int(class_mask.sum()),
                        "mean_confidence": float(confs[class_mask].mean()),
                        "retina_masks": bool(retina_masks),
                        "imgsz": int(imgsz),
                        "mask_available": bool(mask_available),
                    })

    finally:
        progress.close()
        cap.release()
        writer.release()

    elapsed = time.perf_counter() - t_global
    stats_df = pd.DataFrame(stats_rows)

    print("Modelo:", MODEL_NAME)
    print("Video entrada:", video_path)
    print("Video salida raw:", output_path)
    print("Frames leídos:", read_frames)
    print("Frames procesados:", processed_frames)
    print("Tiempo total segundos:", round(elapsed, 2))
    print("FPS end-to-end aprox:", round(processed_frames / max(elapsed, 1e-9), 2))
    print("Máscaras retina:", retina_masks)
    print("IMG_SIZE:", imgsz)

    return output_path, stats_df

## 10. Ejecutar segmentación por instancias

Esto genera video raw, video compatible con Colab y CSV con estadísticas.

In [ ]:
raw_output_path = OUTPUT_DIR / f"{MODEL_NAME}_instance_segmentation_raw_imgsz{IMG_SIZE}.mp4"
browser_output_path = OUTPUT_DIR / f"{MODEL_NAME}_instance_segmentation_browser_imgsz{IMG_SIZE}.mp4"
stats_output_path = OUTPUT_DIR / f"{MODEL_NAME}_instance_segmentation_stats_imgsz{IMG_SIZE}.csv"

raw_output_path, stats_df = process_yolo11_seg_video(
    model=model,
    video_path=VIDEO_PATH,
    output_path=raw_output_path,
    imgsz=IMG_SIZE,
    conf=CONF,
    iou=IOU,
    retina_masks=RETINA_MASKS,
    device=DEVICE,
    frame_stride=FRAME_STRIDE,
    max_frames=MAX_FRAMES,
    draw_info=True,
    collect_stats=True,
)

stats_df.to_csv(stats_output_path, index=False)

browser_output_path = convert_to_browser_mp4(
    input_path=raw_output_path,
    output_path=browser_output_path,
)

print("Video compatible:", browser_output_path)
print("CSV:", stats_output_path)

stats_df.head()

Procesando 27091-361827476_medium.mp4:   0%|          | 0/311 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 259ms
Prepared 1 package in 61ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Modelo: yolo11n-seg
Video entrada: 27091-361827476_medium.mp4
Video salida raw: /content/yolo11_seg_static_video_outputs/yolo11n-seg_instance_segmentation_raw_imgsz1280.mp4
Frames leídos: 311
Frames procesados: 311
Tiempo total segundos: 42.58
FPS end-to-end aprox: 7.3
Máscaras retina: True
IMG_SIZE: 1280
Video compatible: /content/yolo11_seg_static_video_outputs/yolo11n-seg_instance_segmentation_browser_imgsz1280.mp4
CSV: /content/yolo11_seg_static_video_outputs/yolo11n-seg_instance_segmentation_stats_imgsz1280.csv


,model,source_video,read_frame_index,processed_frame_index,class_id,class_name,count,mean_confidence,retina_masks,imgsz,mask_available
0,yolo11n-seg,27091-361827476_medium.mp4,0,0,0,person,8,0.615993,True,1280,True
1,yolo11n-seg,27091-361827476_medium.mp4,0,0,56,chair,8,0.604770,True,1280,True
2,yolo11n-seg,27091-361827476_medium.mp4,0,0,60,dining table,1,0.391379,True,1280,True
3,yolo11n-seg,27091-361827476_medium.mp4,0,0,63,laptop,3,0.565863,True,1280,True
4,yolo11n-seg,27091-361827476_medium.mp4,1,1,0,person,8,0.617678,True,1280,True


## 11. Ver video resultado

In [ ]:
display_video(browser_output_path, width=720)

## 12. Resumen de instancias detectadas por clase

In [ ]:
if not stats_df.empty:
    summary_df = (
        stats_df
        .groupby("class_name", as_index=False)
        .agg(
            total_count=("count", "sum"),
            frames_with_class=("processed_frame_index", "nunique"),
            mean_confidence=("mean_confidence", "mean"),
        )
        .sort_values("total_count", ascending=False)
    )
    print(summary_df)
else:
    print("No hubo detecciones/segmentaciones con el umbral actual.")

     class_name  total_count  frames_with_class  mean_confidence
1         chair         2988                311         0.640312
6        person         2797                311         0.621288
5        laptop         1383                311         0.640562
3  dining table          267                267         0.453758
7            tv          211                210         0.510739
2           cup           88                 88         0.405772
0        bottle           41                 41         0.412993
4      keyboard            1                  1         0.357414


## 13. Descargar resultados

In [ ]:
download_file(browser_output_path)
download_file(stats_output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>